### Loading Dataset

In [1]:
import pandas as pd

In [2]:
df= pd.read_csv(r"C:\Users\Utsab\OneDrive\Desktop\semantic-search-engine\data\raw\tmdb_5000_movies.csv")
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
df.shape

(4803, 20)

In [5]:
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

## Handling missing Values only of required columns

In [6]:
df[['overview', 'tagline', 'genres', 'keywords']].isnull().sum()

overview      3
tagline     844
genres        0
keywords      0
dtype: int64

In [7]:
# dropping columns where overview is null
df = df.dropna(subset=['overview'])

In [8]:
df.shape

(4800, 20)

In [9]:
df['tagline'] = df['tagline'].fillna('')

In [12]:
df['genres'] = df['genres'].fillna('[]')
df['keywords'] = df['keywords'].fillna('[]')

In [13]:
df[['overview', 'tagline', 'genres', 'keywords']].isnull().sum() #no more nulls

overview    0
tagline     0
genres      0
keywords    0
dtype: int64

## Parsing Genres and Keywords

In [15]:
import ast

In [16]:
def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return ' '.join([item['name'] for item in items])
    except:
        return ''

In [17]:
df['genres_clean'] = df['genres'].apply(extract_names)
df['keywords_clean'] = df['keywords'].apply(extract_names)

In [18]:
df[['genres', 'genres_clean', 'keywords_clean']].head()

,genres,genres_clean,keywords_clean
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",Adventure Fantasy Action,ocean drug abuse exotic island east india trad...
2,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...
3,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...
4,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",Action Adventure Science Fiction,based on novel mars medallion space travel pri...


##### Here I obtained cleaner version of genres and keywords

## Creating the column which stores semantic texts

In [23]:
df['semantic_text'] = (
    "Title: " + df['title'] + ". " +
    "Tagline: " + df['tagline'] + ". " +
    "Genres: " + df['genres_clean'] + ". " +
    "Keywords: " + df['keywords_clean'] + ". " +
    "Overview: " + df['overview']
)

In [20]:
df['semantic_text'].iloc[0]

'Title: Avatar. Tagline: Enter the World of Pandora.. Genres: Action Adventure Fantasy Science Fiction. Keywords: culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d. Overview: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

## Creating and Saving final dataset

In [21]:
final_df = df[['id', 'title', 'semantic_text', 'vote_average']].copy()
final_df.reset_index(drop=True, inplace=True)

In [22]:
final_df.to_csv("../data/processed/movies_semantic.csv", index=False)

##### Now the data is ready for further steps